# Reviewer 2, major comments: all five experiments complete
 
Runs completed 12 September 2026. Four studies, 14,911 RASR evaluations in
total. Every number below is reproducible from the CSVs and the two analysis
scripts.
 
| Study | Purpose | Evaluations |
|---|---|---|
| realizations | M1, M3 | 6,020 |
| sweep | M4 | 8,400 |
| realworld | M2 | 400 |
| scale | M5 | 150 |
 
---
 
## M1 and M3, recap
 
The four-way ordering Scale-Free < Real-World < Small-World < Random holds in
**0%** of 100 realizations at ASR = 100%, and never above 31% at any ASR level.
The submitted BA instance was at the 97th percentile of realizations for degree
heterogeneity and the 0th for RASR. What survives is the separation of
hub-dominated from homogeneous networks, in 100% of realizations at ASR = 100%
and 91% or more elsewhere, recovered independently by the deterministic ANC
analysis. Full detail in `FINDINGS_realizations.md`.
 
---
 
## M4, parameter sensitivity: the model family is not the determinant
 
Thirty realizations at each of 28 parameter settings, at ASR = 100%.
 
| Family | span of mean RASR across its parameter grid |
|---|---|
| BA, m = 1 to 5 | 6.62 to 32.98 |
| ER, p = 0.08 to 0.25 | 17.17 to 37.56 |
| WS, k = 4 to 8, p = 0.01 to 1.0 | 21.71 to 38.54 |
 
The families overlap almost completely. A dense scale-free network (BA, m = 5,
RASR 32.98) is far more robust than a sparse random one (ER, p = 0.08, RASR
17.17). Whatever the submitted paper was measuring, it was not a property of
the model family, because the family label does not fix the answer to within
half its range.
 
**The small-world to random crossover is clean.** At k = 4, as the rewiring
probability rises the network loses clustering and converges on the random
model exactly as expected:
 
| WS rewiring p | RASR | clustering |
|---|---|---|
| 0.01 | 22.35 | 0.486 |
| 0.1 | 21.71 | 0.384 |
| 0.3 | 24.79 | 0.234 |
| 1.0 | 25.47 | 0.100 |
| ER p = 0.13 reference | 25.42 | 0.123 |
 
At p = 1.0 the small-world network is statistically indistinguishable from the
random one. The submitted paper's WS < ER claim was measured at p = 0.3, one
point on this curve, and it is inside the noise there.
 
---
 
## The claim that replaces the ordering
 
Pooling every N = 34 instance from the realization and sweep studies,
**n = 1,044**, and predicting RASR at ASR = 100%:
 
| Predictors | R² |
|---|---|
| model label only (BA / WS / ER / Karate) | 0.346 |
| mean degree only | 0.781 |
| mean degree + heterogeneity | 0.881 |
| **mean degree + heterogeneity + clustering** | **0.935** |
| structure + model label | 0.946 |
 
Once the three structural quantities are known, the model label adds
**+0.011** of R². The taxonomy is very nearly redundant.
 
Fitted relationship:
 
```
RASR = 18.40 + 3.44*<k> - 3.14*(k_max/<k>) - 12.70*clustering
```
 
Partial correlations, each controlling for the other two:
 
| Quantity | partial r | p |
|---|---|---|
| mean degree | +0.936 | ~0 |
| degree heterogeneity | -0.817 | 3e-251 |
| clustering | -0.674 | 5e-139 |
 
Variance inflation factors are all near 1.4, so the three effects are
separable rather than an artifact of collinearity.
 
The reading: denser networks are more robust, and at a given density a more
hub-dominated or more clustered network is more fragile. Clustering matters
here because HBnnsAGP targets bridges, and a strongly clustered network has
well-defined communities joined by few of them, which is exactly the structure
that bridge targeting exploits.
 
One caution for whoever writes this up: a simple correlation inside a narrow
density band gives heterogeneity the *wrong sign*, because heterogeneity and
clustering are themselves correlated at r = -0.40. Only the partial correlation
or the multivariate fit gives the right answer. That trap is worth a sentence
in the paper.
 
---
 
## M2, real-world validation: four networks instead of one
 
| Network | N | M | heterogeneity | clustering | RASR @ 100% | RASR @ random ASR |
|---|---|---|---|---|---|---|
| Florentine families | 15 | 20 | 2.25 | 0.160 | 19.17 | 31.98 ± 4.18 |
| Davis southern women | 32 | 89 | 2.52 | 0.000 | 19.89 | 42.39 ± 4.32 |
| Zachary karate club | 34 | 78 | 3.71 | 0.571 | 12.86 | 31.15 ± 4.53 |
| Les Miserables | 77 | 254 | 5.46 | 0.573 | 9.89 | 25.05 ± 3.01 |
 
RASR falls monotonically as heterogeneity rises, across all four empirical
networks. The mechanism established on synthetic graphs holds on real ones.
 
Two caveats to state in the paper. These four are the empirical social networks
bundled with NetworkX, chosen so the results reproduce with no external data
download; larger real networks (infrastructure, transport, collaboration) would
need datasets fetched separately. And they differ in size, so the comparison
between them is not size-controlled the way the synthetic comparison is.
 
The random-ASR column is the answer to Reviewer 2's minor comment 3: it is a
spread over ten independent heterogeneous draws. Karate gives 31.15 ± 4.53 here
against the single draw of 28.24 in the submitted manuscript, confirming that
reporting one draw was not safe.
 
---
 
## M5, scale: the probabilistic result holds, and strengthens
 
RASR under HBnnsAGP, mean degree held near 4 at every size so density does not
drift:
 
| N | BA | WS | ER | BA heterogeneity |
|---|---|---|---|---|
| 34 | 16.80 | 24.76 | 25.14 | 3.64 |
| 100 | 12.46 | 21.69 | 21.95 | 6.81 |
| 200 | 11.04 | 20.82 | 21.35 | 9.14 |
| 500 | 10.22 | 21.91 | 21.77 | 13.96 |
 
The scale-free network separates from both homogeneous models at every size,
and the gap widens as N grows, because BA heterogeneity grows with N while the
other two stay flat. WS and ER remain indistinguishable at every size, which
reinforces the two-group reading rather than the four-way one.
 
This matters for the claim the reviewer objected to. The submitted paper
supported its large-N statement with the deterministic ANC analysis only. The
probabilistic RASR framework now reaches N = 500 directly, so the claim can be
made in the framework the paper is actually about.
 
**The paper's own limitation is wrong and should be corrected.** Section 6.6
states that exhaustive BCnns limits HBnnsAGP to N ≤ 200. Measured cost of a
full evaluation on two cores: 0.3s at N = 34, 3.6s at N = 200, 21s at N = 500,
54s at N = 800. There is no barrier at 200.
 
---
 
## Status
 
All five major comments now have their evidence. None of it is written into the
manuscript yet, which is the next decision.
 
Files: `results_realizations_N34.csv`, `results_sweep_N34.csv`,
`results_realworld_N34.csv`, `results_scale_N{100,200,500}.csv`, `runner.py`,
`analyse_realizations.py`, `analyse_major.py`, `major_output.txt`.